# Agentic Image Enhancement with Strands Agents

This notebook replaces the two-phase pipeline (`assess → compare`) with a genuine agentic loop
powered by [Strands Agents](https://strandsagents.com). The agent has tools to enhance, compare,
reset, and finish — and it decides when to stop based on comparison feedback.

Max iterations: 2 (configurable via `MAX_ITERATIONS`).


## 1. Setup & Configuration

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.insert(0, ".")

In [ ]:
import os
import json
import logging
from pathlib import Path
from typing import Dict, Any

import cv2
import numpy as np
from tools.enhancement_tools import (
    load_image,
    image_to_base64,
    save_image,
    analyze_image,
    execute_operations,
    OPERATIONS,
)
from utils.visualization_utilities import VisualizationUtilities

import re
from dotenv import load_dotenv

logging.basicConfig(level=logging.WARNING, format="%(levelname)s | %(message)s")
logger = logging.getLogger(__name__)

class SensitiveDataFilter(logging.Filter):
    def filter(self, record):
        record.msg = re.sub(r'AKIA[0-9A-Z]{16}', '<REDACTED_KEY>', str(record.msg))
        return True

logger.addFilter(SensitiveDataFilter())

if not load_dotenv():
    logger.warning("No .env file found, using system credentials")

# ── AWS credentials ──
import boto3

AWS_PROFILE = os.environ.get("AWS_PROFILE")
if not AWS_PROFILE:
    logger.warning("AWS_PROFILE not set — boto3 will use default credentials.")

AWS_REGION = os.environ.get("AWS_REGION")
if not AWS_REGION:
    _session = boto3.Session(profile_name=AWS_PROFILE if AWS_PROFILE else None)
    AWS_REGION = _session.region_name
    if not AWS_REGION:
        raise ValueError(
            "AWS_REGION could not be resolved. Set it in .env, environment, "
            "or configure a default region in your AWS profile."
        )
    logger.warning(f"AWS_REGION not set — resolved '{AWS_REGION}' from boto3 session.")

logger.warning(f"Available operations: {list(OPERATIONS.keys())}")

In [ ]:
# ── Model Configuration ──
VISION_MODEL = "us.anthropic.claude-sonnet-4-6"

# ── Enhancement defaults ──
MAX_IMAGE_DIMENSION = 4000
JPEG_QUALITY = 85
OUTPUT_QUALITY = 95
MAX_ITERATIONS = 2

# ── Paths ──
WORKING_DIR = Path(".").resolve()
OUTPUT_DIR = (WORKING_DIR / "outputs/enhanced_output").resolve()

# Ensure output directory is within working directory
if not str(OUTPUT_DIR).startswith(str(WORKING_DIR)):
    raise ValueError("Output directory must be within working directory")

try:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
except PermissionError:
    raise RuntimeError(f"No write permission for {OUTPUT_DIR}")

profile_masked = f"{AWS_PROFILE[:3]}***" if AWS_PROFILE and len(AWS_PROFILE) > 3 else "(default)"
logger.warning(f"Model: {VISION_MODEL}")
logger.warning(f"Region: {AWS_REGION}")
logger.warning(f"Profile: {profile_masked}")
logger.warning(f"Max iterations: {MAX_ITERATIONS}")

## 2. Image State Manager

Holds the original and current image across tool calls so the agent's tools
can read/write shared state without globals.

In [ ]:
class ImageState:
    """Manages image state across agent tool invocations."""

    def __init__(self, image: np.ndarray):
        self.original = image.copy()
        self.current = image.copy()
        self.history: list[Dict[str, Any]] = []
        self.iteration = 0
        self.finished = False
        self.winner = "original"
        self.final_comparison: Dict[str, Any] | None = None

    def reset(self):
        """Reset current image back to original."""
        self.current = self.original.copy()

    def record_iteration(self, operations: list, comparison: Dict[str, Any]):
        self.history.append({
            "iteration": self.iteration,
            "operations": operations,
            "comparison": comparison,
        })
        self.iteration += 1

logger.warning("ImageState defined ✓")

## 3. Strands Agent Tools

Four tools the agent can call:
1. `enhance_image` — apply operations to the current image
2. `compare_with_original` — compare current vs original, get structured feedback
3. `reset_to_original` — undo everything, start fresh
4. `finish_enhancement` — declare done, pick the winner

In [ ]:
from strands import tool


# Rich schema so the LLM knows the exact structure of each operation
ENHANCE_INPUT_SCHEMA = {
    "json": {
        "type": "object",
        "properties": {
            "operations": {
                "type": "array",
                "description": "Ordered list of enhancement operations to apply.",
                "items": {
                    "type": "object",
                    "properties": {
                        "op": {
                            "type": "string",
                            "enum": list(OPERATIONS.keys()),
                            "description": "Operation name.",
                        },
                        "intensity": {
                            "type": "number",
                            "minimum": 0.0,
                            "maximum": 1.0,
                            "description": "Effect strength. 0.0=none, 0.5=moderate, 1.0=max. For brightness: 0.0=darken, 0.5=no change, 1.0=brighten.",
                        },
                        "region": {
                            "type": "object",
                            "description": "Optional region (normalized 0-1 coords). Omit for full image.",
                            "properties": {
                                "x1": {"type": "number", "minimum": 0, "maximum": 1},
                                "y1": {"type": "number", "minimum": 0, "maximum": 1},
                                "x2": {"type": "number", "minimum": 0, "maximum": 1},
                                "y2": {"type": "number", "minimum": 0, "maximum": 1},
                            },
                        },
                    },
                    "required": ["op", "intensity"],
                },
            },
            "reasoning": {
                "type": "string",
                "description": "Brief explanation of why these operations were chosen.",
            },
        },
        "required": ["operations", "reasoning"],
    }
}


def create_tools(state: ImageState):
    """Create tool functions bound to a specific ImageState instance."""

    @tool(name="enhance_image", description="Apply targeted image enhancement operations to improve document readability. Operations are applied sequentially.", inputSchema=ENHANCE_INPUT_SCHEMA)
    def enhance_image(operations: list, reasoning: str) -> str:
        """Apply enhancement operations to the current image."""
        logger.warning(f"enhance_image called: {reasoning}")
        enhanced, op_log = execute_operations(state.current, operations)
        state.current = enhanced

        VisualizationUtilities.show_operations_log(op_log)

        return json.dumps({
            "status": "success",
            "operations_applied": len(op_log),
            "reasoning": reasoning,
            "note": "Call compare_with_original to evaluate the result.",
        })

    @tool
    def compare_with_original() -> str:
        """Compare the current enhanced image against the original.

        Returns quality metrics for both versions so you can judge
        whether the enhancement helped. Call this after enhance_image.
        """
        orig_metrics = analyze_image(state.original)
        curr_metrics = analyze_image(state.current)

        # Compute deltas
        deltas = {}
        for key in orig_metrics:
            if isinstance(orig_metrics[key], (int, float)):
                deltas[key] = round(curr_metrics[key] - orig_metrics[key], 3)

        state.record_iteration([], {"original": orig_metrics, "enhanced": curr_metrics, "deltas": deltas})

        return json.dumps({
            "original_metrics": orig_metrics,
            "enhanced_metrics": curr_metrics,
            "deltas": deltas,
            "iteration": state.iteration,
            "note": (
                "Positive contrast_std delta = more contrast. "
                "Positive sharpness_laplacian delta = sharper. "
                "If enhancement helped, call finish_enhancement with winner='enhanced'. "
                "If it made things worse, call reset_to_original and try different operations, "
                "or call finish_enhancement with winner='original'."
            ),
        })

    @tool
    def reset_to_original(reasoning: str) -> str:
        """Reset the image back to the original, discarding all enhancements.

        Use this when the previous enhancement made things worse and you
        want to try a different approach.

        Args:
            reasoning: Why you're resetting.
        """
        state.reset()
        logger.warning(f"Reset to original: {reasoning}")
        return json.dumps({"status": "reset", "reasoning": reasoning})

    @tool
    def finish_enhancement(winner: str, reasoning: str) -> str:
        """Declare the enhancement process complete.

        Call this when you're satisfied with the result OR when you've
        determined the original is better than any enhancement.

        Args:
            winner: Either 'original' or 'enhanced'.
            reasoning: Why this version was chosen as the winner.
        """
        state.finished = True
        state.winner = winner
        state.final_comparison = {"winner": winner, "reasoning": reasoning}
        logger.warning(f"Finished: winner={winner} — {reasoning}")
        return json.dumps({"status": "done", "winner": winner, "reasoning": reasoning})

    return [enhance_image, compare_with_original, reset_to_original, finish_enhancement]

logger.warning("Tools defined ✓")

## 4. System Prompt

Combines assessment guidance, operation knowledge, and the iterative refinement instructions.

In [ ]:
SYSTEM_PROMPT = f"""
You are an expert document image analyst. You enhance document images for
downstream text extraction by a vision LLM.

You have {MAX_ITERATIONS} iterations maximum to get the best result.

<workflow>
1. Examine the image carefully.
2. If it's already good enough, call finish_enhancement with winner="original".
3. Otherwise, call enhance_image with targeted operations.
4. Call compare_with_original to evaluate your enhancement.
5. Based on the comparison:
   - If metrics improved: call finish_enhancement with winner="enhanced".
   - If metrics got worse: call reset_to_original, then try different
     operations with gentler intensities.
   - If marginal: you may try one more iteration or accept the result.
6. After {MAX_ITERATIONS} iterations, you MUST call finish_enhancement.
</workflow>

<operation_guidance>
Available operations: {', '.join(OPERATIONS.keys())}

- Order matters: deskew/crop FIRST, then denoise, then contrast/brightness, sharpen LAST.
- Be conservative: 0.3-0.5 intensity is usually enough. Only go above 0.7 for severe degradation.
- Use region targeting if only part of the image has issues.
- For brightness: 0.0=darken, 0.5=no change, 1.0=brighten.
- Manuscripts need gentle treatment. Sheet music/diagrams: prefer contrast over sharpening.
</operation_guidance>

<rules>
- ALWAYS call compare_with_original after enhance_image.
- ALWAYS end by calling finish_enhancement.
- Do NOT enhance images that are already clear and well-oriented.
- If you reset, try a DIFFERENT approach — don't repeat what failed.
</rules>
"""

logger.warning("System prompt defined ✓")

## 5. Enhancement Utilities

Helper class that wraps the Strands agent invocation, following the same
utility class pattern as `VisualizationUtilities`.

In [ ]:
import boto3
from strands import Agent
from strands.models import BedrockModel


class EnhancementUtilities:
    """Orchestrates the Strands-based agentic enhancement loop."""

    @staticmethod
    def _resize_for_llm(image: np.ndarray, max_dim: int = MAX_IMAGE_DIMENSION) -> np.ndarray:
        h, w = image.shape[:2]
        if max(h, w) <= max_dim:
            return image
        scale = max_dim / max(h, w)
        new_w, new_h = int(w * scale), int(h * scale)
        return cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_AREA)

    @staticmethod
    def _build_image_message(image: np.ndarray, context: str = "") -> list:
        """Build a multimodal message with the image for the Strands agent."""
        resized = EnhancementUtilities._resize_for_llm(image)
        b64 = image_to_base64(resized, fmt="jpeg", quality=JPEG_QUALITY)
        import base64 as b64_mod
        image_bytes = b64_mod.b64decode(b64)

        blocks = [
            {
                "image": {
                    "format": "jpeg",
                    "source": {"bytes": image_bytes},
                }
            },
            {"text": "Analyze this document image and enhance it for text extraction."},
        ]

        if context:
            blocks.append({"text": f"Document context: {context}"})

        return blocks

    @staticmethod
    def create_agent(state: ImageState) -> Agent:
        """Create a Strands agent configured for image enhancement."""
        session = boto3.Session(
            region_name=AWS_REGION,
            profile_name=AWS_PROFILE if AWS_PROFILE else None,
        )

        model = BedrockModel(
            model_id=VISION_MODEL,
            boto_session=session,
            max_tokens=2048,
        )

        tools = create_tools(state)

        return Agent(
            model=model,
            system_prompt=SYSTEM_PROMPT,
            tools=tools,
        )

    @staticmethod
    def enhance(
        image_source: str | Path | np.ndarray,
        context: str = "",
        save_output: bool = True,
    ) -> Dict[str, Any]:
        """
        Run the full agentic enhancement loop.

        Args:
            image_source: Path to image file or numpy array.
            context: Optional document context (e.g., "18th century manuscript").
            save_output: Whether to save the winner to OUTPUT_DIR.

        Returns:
            Dict with original, enhanced, winner, history, and metadata.
        """
        image = load_image(image_source)
        source_name = Path(image_source).stem if isinstance(image_source, (str, Path)) else "image"

        logger.warning('=' * 60)
        logger.warning("  STRANDS AGENTIC IMAGE ENHANCEMENT")
        logger.warning(f"  Source: {source_name}")
        logger.warning(f"  Dimensions: {image.shape[1]}x{image.shape[0]}")
        logger.warning(f"  Model: {VISION_MODEL}")
        logger.warning(f"  Max iterations: {MAX_ITERATIONS}")
        logger.warning('=' * 60)

        # Set up state and agent
        state = ImageState(image)
        agent = EnhancementUtilities.create_agent(state)

        # Build the multimodal prompt
        message = EnhancementUtilities._build_image_message(image, context)

        # Run the agent
        logger.warning("Running Strands agent...")
        response = agent(message)

        # Collect results
        winner = state.winner if state.finished else "original"
        winner_image = state.current if winner == "enhanced" else state.original

        result = {
            "original": state.original,
            "enhanced": state.current if winner == "enhanced" else None,
            "winner": winner,
            "winner_image": winner_image,
            "history": state.history,
            "final_comparison": state.final_comparison,
            "iterations": state.iteration,
        }

        # Display
        if winner == "enhanced":
            logger.warning(f"Winner: ENHANCED (after {state.iteration} iteration(s))")
            VisualizationUtilities.show_comparison(
                state.original, state.current,
                title=f"{source_name} — Strands Agentic Enhancement",
            )
        else:
            logger.warning("Winner: ORIGINAL")
            VisualizationUtilities.show_image(
                state.original,
                title=f"{source_name} — Original (no enhancement needed)",
            )

        if state.final_comparison:
            logger.warning(f"Reasoning: {state.final_comparison.get('reasoning', 'N/A')}")

        # Save
        if save_output:
            path = save_image(
                winner_image,
                OUTPUT_DIR / f"{source_name}_{winner}.jpg",
                OUTPUT_QUALITY,
            )
            result["saved_path"] = path
            logger.warning(f"Saved: {path}")

        return result

logger.warning("EnhancementUtilities defined ✓")

---
## 6. Try It Out

In [ ]:
import ipywidgets as widgets
from pathlib import Path

input_dir = Path("inputs")
image_files = sorted(
    [
        f.name
        for f in input_dir.iterdir()
        if f.suffix.lower() in (".jpg", ".jpeg", ".png", ".tiff", ".tif", ".bmp")
    ]
)

dropdown = widgets.Dropdown(
    options=image_files, description="Image:", layout=widgets.Layout(width="400px")
)

IMAGE_PATH = str(input_dir / dropdown.value)


def on_change(change):
    global IMAGE_PATH
    IMAGE_PATH = str(input_dir / change["new"])

context_options = [
    "Circulation Ledger",
    "Ancient Chinese Book",
    "Sheet Music",
    "18th Century Manuscript",
    "Technical Diagram",
]

context_dropdown = widgets.Dropdown(
    options=context_options,
    description="Context:",
    layout=widgets.Layout(width="400px"),
)

CONTEXT = context_dropdown.value


def on_context_change(change):
    global CONTEXT
    CONTEXT = change["new"]


context_dropdown.observe(on_context_change, names="value")
dropdown.observe(on_change, names="value")

logger.warning("Select and match the context to the image below:")
display(context_dropdown)

display(dropdown)

In [ ]:
result = EnhancementUtilities.enhance(IMAGE_PATH, context=CONTEXT)